# 07 · Dataset v2 and the 2024 distribution shift

Switch to the Grid-Sentinel feed (Grid-India daily Power Supply Position reports, 2019–2026), validate it, and define the final protocol: **train 2019–2023 · validate 2024 · external test 2025–2026**.

**Key results**
- Cross-checked against an official Grid-India PDF (exact match) and against dataset v1 (1,796 of 1,811 overlapping days within 0.1 MU; two zero-valued days removed).
- 617 external origins have a complete 56-day history.
- XGBoost v2 **loses to weekly naive on 2024** (9.05 vs 8.45 MU): 2024 summer peaks (177.7 MU) exceed anything in training (153.5 MU) and trees cannot extrapolate (max prediction 145.5). Adding observed weather made it worse (10.08); residual XGBoost ≈ 8.60.
- Zero-shot Chronos-Bolt (7.18) and TimesFM (7.16) are statistically indistinguishable on 2024 (bootstrap 95% CI of the difference: −0.56 to +0.57 MU).

> Paths are relative to `notebooks/`. Outputs are from the original research run; see [`data/README.md`](../data/README.md) for data that must be downloaded.

➡ Next: [08 · Chronos-2 LoRA fine-tuning and weather covariates](08_chronos2_lora_finetuning.ipynb)

In [1]:
import pandas as pd

url = (
    "https://raw.githubusercontent.com/"
    "Deducinnamoroll/Grid-Sentinel/main/"
    "Dataset/study3_states.csv"
)

df = pd.read_csv(url)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print(df.head(3).to_string(index=False))

delhi = df[
    df["state"].astype(str).str.strip().str.upper() == "DELHI"
].copy()

delhi["date"] = pd.to_datetime(delhi["date"])

delhi = delhi[
    delhi["date"].between("2025-01-01", "2026-11-18")
].sort_values("date")

print("\nDelhi rows:", len(delhi))
print("Date range:", delhi["date"].min(), delhi["date"].max())
print("Duplicate dates:", delhi["date"].duplicated().sum())
print("Missing energy values:", delhi["energy_met_mu"].isna().sum())

display(delhi.head())

Shape: (104539, 10)
Columns: ['date', 'region', 'state', 'max_demand_met_mw', 'shortage_max_demand_mw', 'energy_met_mu', 'drawal_schedule_mu', 'od_ud_mu', 'max_od_mw', 'energy_shortage_mu']
      date region     state  max_demand_met_mw  shortage_max_demand_mw  energy_met_mu  drawal_schedule_mu  od_ud_mu  max_od_mw  energy_shortage_mu
2018-12-31     ER     Bihar             4362.0                     0.0           76.8                70.7       0.7      460.0                 0.0
2018-12-31     ER       DVC             3123.0                     0.0           65.3               -31.4       2.2      396.0                 0.0
2018-12-31     ER Jharkhand             1199.0                     0.0           26.1                19.0       0.8      169.0                 0.0

Delhi rows: 685
Date range: 2025-01-01 00:00:00 2026-11-18 00:00:00
Duplicate dates: 0
Missing energy values: 0


,date,region,state,max_demand_met_mw,shortage_max_demand_mw,energy_met_mu,drawal_schedule_mu,od_ud_mu,max_od_mw,energy_shortage_mu
77617,2025-01-01,NR,Delhi,5519.0,0.0,87.43,85.50,-0.38,404.0,0.0
77656,2025-01-02,NR,Delhi,5155.0,0.0,86.74,85.32,-1.22,368.0,0.0
77695,2025-01-03,NR,Delhi,5611.0,0.0,89.69,88.03,-0.75,475.0,0.0
77734,2025-01-04,NR,Delhi,5065.0,0.0,84.23,82.55,-0.96,405.0,0.0
77773,2025-01-05,NR,Delhi,5115.0,0.0,81.57,80.16,-1.22,379.0,0.0


In [2]:
# Calendar completeness
calendar = pd.date_range(
    "2025-01-01", "2026-11-18", freq="D"
)

missing_dates = calendar.difference(delhi["date"])

print("Expected calendar days:", len(calendar))
print("Actual records:", len(delhi))
print("Missing calendar dates:", missing_dates.tolist())

# Official NRLDC report validation
official = delhi.loc[
    delhi["date"] == "2026-11-17",
    ["date", "energy_met_mu", "max_demand_met_mw"]
]

print("\nOfficial report cross-check:")
display(official)

if len(official) == 1:
    record = official.iloc[0]
    print(
        "Energy matches official PDF:",
        abs(record["energy_met_mu"] - 78.62) < 0.01
    )
    print(
        "Peak matches official PDF:",
        abs(record["max_demand_met_mw"] - 4081) < 1
    )

# Compare against published monthly averages
monthly = (
    delhi.assign(month=delhi["date"].dt.to_period("M"))
    .groupby("month")
    .agg(
        avg_energy_mu=("energy_met_mu", "mean"),
        days=("energy_met_mu", "size"),
    )
)

print("\nMonthly energy averages:")
display(monthly.loc["2025-10":"2026-10"].round(2))

Expected calendar days: 687
Actual records: 685
Missing calendar dates: [Timestamp('2025-05-22 00:00:00'), Timestamp('2025-05-23 00:00:00')]

Official report cross-check:


,date,energy_met_mu,max_demand_met_mw
104472,2026-11-17,78.62,4081.0


Energy matches official PDF: True
Peak matches official PDF: True

Monthly energy averages:


,avg_energy_mu,days
month,,
2025-10,91.89,31
2025-11,75.44,30
2025-12,80.82,31
2026-01,90.60,31
2026-02,79.29,28
2026-03,82.64,31
2026-04,106.18,30
2026-05,135.47,31
2026-06,144.71,30


In [3]:
delhi_all = df.loc[
    df["state"].astype(str).str.strip().str.upper() == "DELHI"
].copy()

delhi_all["date"] = pd.to_datetime(delhi_all["date"])
delhi_all = delhi_all.sort_values("date")

print("First date:", delhi_all["date"].min())
print("Last date:", delhi_all["date"].max())
print("Total Delhi observations:", len(delhi_all))
print("Duplicate dates:", delhi_all["date"].duplicated().sum())

coverage = (
    delhi_all.groupby(delhi_all["date"].dt.year)
    .agg(
        observations=("date", "size"),
        missing_energy=("energy_met_mu", lambda x: x.isna().sum()),
        min_date=("date", "min"),
        max_date=("date", "max"),
    )
)

display(coverage)

First date: 2018-12-31 00:00:00
Last date: 2026-11-18 00:00:00
Total Delhi observations: 2810
Duplicate dates: 0


,observations,missing_energy,min_date,max_date
date,,,,
2018,1,0,2018-12-31,2018-12-31
2019,357,0,2019-01-01,2019-12-31
2020,320,0,2020-01-01,2020-12-31
2021,360,0,2021-01-01,2021-12-31
2022,356,0,2022-01-01,2022-12-31
2023,365,0,2023-01-01,2023-12-31
2024,366,0,2024-01-01,2024-12-31
2025,363,0,2025-01-01,2025-12-31
2026,322,0,2026-01-01,2026-11-18


In [6]:
# Compare Grid-Sentinel with our cleaned Zenodo series.
from pathlib import Path
DATA = Path("../data") 
zenodo = pd.read_csv(
    DATA / "processed/delhi_energy_clean.csv",
    parse_dates=["Date"]
)

zenodo = zenodo.rename(columns={
    "Date": "date",
    "energy_met_MU": "zenodo_mu",
})

github = delhi_all[["date", "energy_met_mu"]].rename(
    columns={"energy_met_mu": "github_mu"}
)

overlap = zenodo[["date", "zenodo_mu"]].merge(
    github,
    on="date",
    how="inner",
    validate="one_to_one"
).dropna()

overlap["difference"] = (
    overlap["github_mu"] - overlap["zenodo_mu"]
)

overlap["abs_difference"] = overlap["difference"].abs()

print("Overlapping valid dates:", len(overlap))
print("Exact matches:", (overlap["abs_difference"] < 0.001).sum())
print("Within 0.1 MU:", (overlap["abs_difference"] <= 0.1).sum())
print("Mean absolute difference:",
      overlap["abs_difference"].mean())
print("Maximum absolute difference:",
      overlap["abs_difference"].max())

print("\nLargest discrepancies:")
display(
    overlap.nlargest(15, "abs_difference")
)

Overlapping valid dates: 1811
Exact matches: 1370
Within 0.1 MU: 1796
Mean absolute difference: 0.10191054665930431
Maximum absolute difference: 62.6

Largest discrepancies:


,date,zenodo_mu,github_mu,difference,abs_difference
315,2019-11-19,62.6,0.0,-62.6,62.6
314,2019-11-18,61.5,0.0,-61.5,61.5
1235,2022-07-19,140.9,129.6,-11.3,11.3
1158,2022-05-03,123.5,116.7,-6.8,6.8
1226,2022-07-10,122.1,116.6,-5.5,5.5
154,2019-06-07,129.2,134.4,5.2,5.2
156,2019-06-09,123.8,129.0,5.2,5.2
26,2019-01-26,58.6,63.7,5.1,5.1
153,2019-06-06,134.4,131.2,-3.2,3.2
58,2019-02-28,66.1,64.1,-2.0,2.0


In [7]:
# 1. Look for zero or negative energy readings in GitHub data
suspicious = delhi_all.loc[
    delhi_all["energy_met_mu"] <= 0,
    ["date", "energy_met_mu", "max_demand_met_mw"]
]

print("Zero/negative observations:", len(suspicious))
display(suspicious)

# 2. Understand the difference distribution by year
overlap["year"] = overlap["date"].dt.year

summary = overlap.groupby("year").agg(
    compared=("difference", "size"),
    mae=("abs_difference", "mean"),
    max_difference=("abs_difference", "max"),
    above_0_1=("abs_difference", lambda x: (x > 0.1).sum()),
    above_1=("abs_difference", lambda x: (x > 1).sum()),
)

display(summary.round(3))

# 3. Investigate dates with material disagreement
disagreements = overlap.loc[
    overlap["abs_difference"] > 1
].sort_values("date")

display(disagreements)

Zero/negative observations: 2


,date,energy_met_mu,max_demand_met_mw
11318,2019-11-18,0.0,NaN
11354,2019-11-19,0.0,NaN


,compared,mae,max_difference,above_0_1,above_1
year,,,,,
2018,1,0.000,0.00,0,0
2019,353,0.412,62.60,9,7
2020,275,0.000,0.00,0,0
2021,347,0.000,0.00,0,0
2022,352,0.077,11.30,6,5
2023,364,0.025,0.05,0,0
2024,119,0.023,0.05,0,0


,date,zenodo_mu,github_mu,difference,abs_difference,year
26,2019-01-26,58.6,63.7,5.1,5.1,2019
58,2019-02-28,66.1,64.1,-2.0,2.0,2019
153,2019-06-06,134.4,131.2,-3.2,3.2,2019
154,2019-06-07,129.2,134.4,5.2,5.2,2019
156,2019-06-09,123.8,129.0,5.2,5.2,2019
314,2019-11-18,61.5,0.0,-61.5,61.5,2019
315,2019-11-19,62.6,0.0,-62.6,62.6,2019
1084,2022-02-16,65.6,64.3,-1.3,1.3,2022
1158,2022-05-03,123.5,116.7,-6.8,6.8,2022
1226,2022-07-10,122.1,116.6,-5.5,5.5,2022


Create the cleaned v2 series

In [8]:
from pathlib import Path
import pandas as pd

OUTPUT = Path("../data/processed")
OUTPUT.mkdir(parents=True, exist_ok=True)

v2 = delhi_all[
    ["date", "energy_met_mu", "max_demand_met_mw"]
].copy()

v2 = v2.set_index("date").sort_index()

calendar = pd.date_range(
    v2.index.min(),
    v2.index.max(),
    freq="D",
    name="date"
)

v2 = v2.reindex(calendar)

# Two anomalous records: zero energy and unavailable peak demand
invalid = (
    v2["energy_met_mu"].eq(0)
    & v2["max_demand_met_mw"].isna()
)

v2.loc[invalid, "energy_met_mu"] = float("nan")

# Keep an audit flag for anomalous/absent records
v2["energy_missing"] = v2["energy_met_mu"].isna()

print("Total calendar days:", len(v2))
print("Valid energy observations:", v2["energy_met_mu"].notna().sum())
print("Missing/invalid energy observations:", v2["energy_missing"].sum())

# Save separate development and external-test datasets
v2.loc[: "2024-12-31"].to_csv(
    OUTPUT / "gridpulse_v2_development.csv"
)

v2.loc["2025-01-01":].to_csv(
    OUTPUT / "gridpulse_v2_external_test.csv"
)

Total calendar days: 2880
Valid energy observations: 2808
Missing/invalid energy observations: 72


Check how many forecast windows survive

In [9]:
energy = v2["energy_met_mu"]
HORIZON = 7

for context in [28, 42, 56, 84, 112]:
    valid = energy.notna().to_numpy()
    window = context + HORIZON

    eligible = sum(
        valid[i:i+window].all()
        for i in range(len(valid) - window + 1)
    )

    print(
        f"Context {context:3d} days: "
        f"{eligible:4d} eligible origins"
    )

Context  28 days: 1985 eligible origins
Context  42 days: 1841 eligible origins
Context  56 days: 1722 eligible origins
Context  84 days: 1498 eligible origins
Context 112 days: 1319 eligible origins


Split-wise eligibility

In [10]:
import numpy as np
import pandas as pd

energy = v2["energy_met_mu"]
valid = energy.notna().to_numpy()

HORIZON = 7
records = []

for context in [28, 56]:
    for i in range(context, len(energy) - HORIZON + 1):
        if not valid[i-context:i+HORIZON].all():
            continue

        first_target = energy.index[i]
        last_target = energy.index[i + HORIZON - 1]

        if last_target < pd.Timestamp("2024-01-01"):
            split = "train"
        elif (
            first_target >= pd.Timestamp("2024-01-01")
            and last_target < pd.Timestamp("2025-01-01")
        ):
            split = "validation"
        elif first_target >= pd.Timestamp("2025-01-01"):
            split = "test"
        else:
            continue

        records.append({
            "context": context,
            "origin": energy.index[i - 1],
            "split": split,
        })

eligibility = pd.DataFrame(records)

display(
    eligibility.groupby(["context", "split"])
    .size()
    .unstack(fill_value=0)
)

split,test,train,validation
context,,,
28,645,968,360
56,617,733,360


Build GridPulse v2's feature matrix

In [11]:
energy = v2["energy_met_mu"]
HORIZON = 7
CONTEXT = 28

rows = []

for i in range(CONTEXT, len(energy) - HORIZON + 1):
    history = energy.iloc[i-CONTEXT:i]
    future = energy.iloc[i:i+HORIZON]

    if history.isna().any() or future.isna().any():
        continue

    first_target = future.index[0]
    last_target = future.index[-1]

    if last_target < pd.Timestamp("2024-01-01"):
        split = "train"
    elif (
        first_target >= pd.Timestamp("2024-01-01")
        and last_target < pd.Timestamp("2025-01-01")
    ):
        split = "validation"
    else:
        continue

    values = history.to_numpy()

    for h in range(HORIZON):
        target_date = future.index[h]

        row = {
            "origin": history.index[-1],
            "target_date": target_date,
            "split": split,
            "horizon": h + 1,
            "target": future.iloc[h],
            "rolling_mean_7": values[-7:].mean(),
            "rolling_std_7": values[-7:].std(),
            "rolling_mean_28": values.mean(),
            "rolling_std_28": values.std(),
            "day_of_week": target_date.dayofweek,
            "month": target_date.month,
            "day_of_year": target_date.dayofyear,
        }

        row.update({
            f"lag_{j}": values[-j]
            for j in range(1, CONTEXT + 1)
        })

        rows.append(row)

features_v2 = pd.DataFrame(rows)

display(
    features_v2.groupby("split").agg(
        origins=("origin", "nunique"),
        forecast_rows=("target", "size")
    )
)

,origins,forecast_rows
split,,
train,968,6776
validation,360,2520


Train XGBoost v2

In [12]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

exclude = {"origin", "target_date", "split", "target"}
feature_cols = [c for c in features_v2.columns if c not in exclude]

train = features_v2[features_v2["split"] == "train"]
val = features_v2[features_v2["split"] == "validation"].copy()

model_v2 = XGBRegressor(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    tree_method="hist",
    device="cpu",
    random_state=42,
    n_jobs=-1,
)

model_v2.fit(train[feature_cols], train["target"])

val["xgb_pred"] = model_v2.predict(val[feature_cols])

# Weekly seasonal naive: repeat the value from 7 days before target.
# Since each row predicts h days ahead, that is lag_(7-h+1).
val["naive_pred"] = [
    row[f"lag_{8 - int(row['horizon'])}"]
    for _, row in val.iterrows()
]

for name in ["naive_pred", "xgb_pred"]:
    print(
        name,
        "MAE:", round(mean_absolute_error(val["target"], val[name]), 3),
        "RMSE:", round(root_mean_squared_error(val["target"], val[name]), 3),
    )

naive_pred MAE: 8.449 RMSE: 11.223
xgb_pred MAE: 9.047 RMSE: 13.13


Where is XGBoost failing?

In [13]:
import numpy as np

# Errors by forecast horizon
horizon_errors = val.groupby("horizon").apply(
    lambda g: pd.Series({
        "naive_mae": np.mean(
            np.abs(g["target"] - g["naive_pred"])
        ),
        "xgb_mae": np.mean(
            np.abs(g["target"] - g["xgb_pred"])
        ),
    }),
    include_groups=False,
)

horizon_errors["xgb_improvement_pct"] = (
    100 *
    (horizon_errors["naive_mae"] - horizon_errors["xgb_mae"])
    / horizon_errors["naive_mae"]
)

print("Errors by horizon:")
display(horizon_errors.round(3))

# Errors by month
monthly_errors = val.assign(
    month=val["target_date"].dt.month,
    naive_error=lambda d: (d["target"] - d["naive_pred"]).abs(),
    xgb_error=lambda d: (d["target"] - d["xgb_pred"]).abs(),
).groupby("month").agg(
    naive_mae=("naive_error", "mean"),
    xgb_mae=("xgb_error", "mean"),
)

monthly_errors["xgb_improvement_pct"] = (
    100 *
    (monthly_errors["naive_mae"] - monthly_errors["xgb_mae"])
    / monthly_errors["naive_mae"]
)

print("\nErrors by month:")
display(monthly_errors.round(3))

Errors by horizon:


,naive_mae,xgb_mae,xgb_improvement_pct
horizon,,,
1,8.496,8.137,4.225
2,8.477,8.645,-1.984
3,8.468,9.056,-6.942
4,8.449,9.250,-9.479
5,8.435,9.334,-10.660
6,8.416,9.402,-11.705
7,8.399,9.507,-13.189



Errors by month:


,naive_mae,xgb_mae,xgb_improvement_pct
month,,,
1,5.089,5.067,0.430
2,4.148,2.301,44.532
3,3.970,4.138,-4.249
4,6.786,5.194,23.471
5,12.791,19.947,-55.946
6,12.445,22.016,-76.908
7,11.474,13.104,-14.212
8,10.189,8.465,16.916
9,12.779,10.390,18.694


Test the summer distribution-shift hypothesis

In [14]:
# Compare training and validation target distributions

print("Training target quantiles:")
display(train["target"].quantile([0, .5, .9, .95, .99, 1]))

print("\n2024 target quantiles:")
display(val["target"].quantile([0, .5, .9, .95, .99, 1]))

# Monthly signed error: actual minus prediction
diagnostics = val.assign(
    month=val["target_date"].dt.month,
    xgb_signed_error=val["target"] - val["xgb_pred"],
    naive_signed_error=val["target"] - val["naive_pred"],
)

monthly_bias = diagnostics.groupby("month").agg(
    actual_mean=("target", "mean"),
    xgb_pred_mean=("xgb_pred", "mean"),
    naive_pred_mean=("naive_pred", "mean"),
    xgb_bias=("xgb_signed_error", "mean"),
    naive_bias=("naive_signed_error", "mean"),
)

display(monthly_bias.round(2))

# Check whether predictions are saturating near their training range
print("\nTraining maximum:", train["target"].max())
print("2024 maximum:", val["target"].max())
print("Maximum XGBoost prediction:", val["xgb_pred"].max())

Training target quantiles:


0.00     44.10
0.50     85.55
0.90    130.40
0.95    136.41
0.99    146.35
1.00    153.50
Name: target, dtype: float64


2024 target quantiles:


0.00     55.650
0.50     98.790
0.90    149.009
0.95    157.357
0.99    163.920
1.00    177.670
Name: target, dtype: float64

,actual_mean,xgb_pred_mean,naive_pred_mean,xgb_bias,naive_bias
month,,,,,
1,88.67,87.470001,87.22,1.20,1.45
2,75.03,74.910004,78.28,0.13,-3.25
3,71.10,68.940002,69.93,2.17,1.17
4,94.76,91.779999,88.13,2.98,6.63
5,135.76,117.820000,123.86,17.94,11.91
6,151.80,130.669998,155.36,21.13,-3.56
7,140.98,131.529999,139.22,9.45,1.76
8,127.55,132.800003,133.32,-5.25,-5.78
9,118.74,115.029999,117.87,3.71,0.87



Training maximum: 153.5
2024 maximum: 177.67
Maximum XGBoost prediction: 145.47137


Get Delhi's historical weather

In [15]:
import requests
import pandas as pd

url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": 28.6139,
    "longitude": 77.2090,
    "start_date": "2019-01-01",
    "end_date": "2024-12-31",
    "daily": ",".join([
        "temperature_2m_max",
        "temperature_2m_min",
        "temperature_2m_mean",
        "relative_humidity_2m_mean",
        "apparent_temperature_max",
    ]),
    "timezone": "Asia/Kolkata",
}

response = requests.get(url, params=params, timeout=60)
response.raise_for_status()

weather = pd.DataFrame(response.json()["daily"])
weather = weather.rename(columns={"time": "date"})
weather["date"] = pd.to_datetime(weather["date"])

print(weather.shape)
display(weather.head())

(2192, 6)


,date,temperature_2m_max,temperature_2m_min,temperature_2m_mean,relative_humidity_2m_mean,apparent_temperature_max
0,2019-01-01,20.2,5.8,12.7,71,18.7
1,2019-01-02,21.1,7.5,14.1,71,19.8
2,2019-01-03,20.7,7.4,13.4,70,17.9
3,2019-01-04,18.8,6.9,12.6,70,17.1
4,2019-01-05,20.5,7.7,13.4,69,17.0


Calculate corelations

In [16]:
combined = (
    val[["target_date", "target", "xgb_pred"]]
    .rename(columns={"target_date": "date"})
    .merge(weather, on="date", validate="many_to_one")
)

combined["xgb_error"] = (
    combined["target"] - combined["xgb_pred"]
)

# One daily record per target date, avoiding
# duplicate observations from overlapping forecasts.
daily = (
    combined.groupby("date", as_index=False)
    .agg(
        energy_mu=("target", "first"),
        temperature_max=("temperature_2m_max", "first"),
        apparent_temp=("apparent_temperature_max", "first"),
        mean_xgb_error=("xgb_error", "mean"),
    )
)

display(
    daily[
        ["energy_mu", "temperature_max",
         "apparent_temp", "mean_xgb_error"]
    ].corr().round(3)
)

,energy_mu,temperature_max,apparent_temp,mean_xgb_error
energy_mu,1.000,0.698,0.803,0.560
temperature_max,0.698,1.000,0.927,0.502
apparent_temp,0.803,0.927,1.000,0.407
mean_xgb_error,0.560,0.502,0.407,1.000


In [17]:
weather_indexed = (
    weather.set_index("date")
    .sort_index()
)

weather_cols = [
    "temperature_2m_max",
    "temperature_2m_mean",
    "relative_humidity_2m_mean",
    "apparent_temperature_max",
]

weather_features = weather_indexed[weather_cols].copy()

# For each forecast origin, use only weather already observed.
# Rolling windows end at the origin date.
for col in weather_cols:
    weather_features[f"{col}_lag1"] = weather_features[col]
    weather_features[f"{col}_mean7"] = (
        weather_features[col].rolling(7, min_periods=7).mean()
    )
    weather_features[f"{col}_mean28"] = (
        weather_features[col].rolling(28, min_periods=28).mean()
    )

origin_weather_cols = [
    c for c in weather_features.columns
    if c.endswith(("_lag1", "_mean7", "_mean28"))
]

features_weather = features_v2.merge(
    weather_features[origin_weather_cols],
    left_on="origin",
    right_index=True,
    how="left",
    validate="many_to_one",
)

print("Weather feature columns:", len(origin_weather_cols))
print(
    "Training missing weather rows:",
    features_weather.loc[
        features_weather["split"] == "train",
        origin_weather_cols
    ].isna().any(axis=1).sum()
)

print(
    "Validation missing weather rows:",
    features_weather.loc[
        features_weather["split"] == "validation",
        origin_weather_cols
    ].isna().any(axis=1).sum()
)

Weather feature columns: 12
Training missing weather rows: 0
Validation missing weather rows: 0


In [18]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

train_w = features_weather[
    features_weather["split"] == "train"
]

val_w = features_weather[
    features_weather["split"] == "validation"
].copy()

weather_feature_cols = feature_cols + origin_weather_cols

model_weather = XGBRegressor(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    tree_method="hist",
    device="cpu",
    random_state=42,
    n_jobs=-1,
)

model_weather.fit(
    train_w[weather_feature_cols],
    train_w["target"]
)

val_w["weather_pred"] = model_weather.predict(
    val_w[weather_feature_cols]
)

print("Weather-aware XGBoost")
print(
    "MAE:",
    round(mean_absolute_error(
        val_w["target"], val_w["weather_pred"]
    ), 3)
)
print(
    "RMSE:",
    round(root_mean_squared_error(
        val_w["target"], val_w["weather_pred"]
    ), 3)
)

print("\nOriginal XGBoost MAE: 9.047")
print("Weekly naive MAE:     8.449")

Weather-aware XGBoost
MAE: 10.082
RMSE: 15.762

Original XGBoost MAE: 9.047
Weekly naive MAE:     8.449


In [19]:
weather_diagnostics = val_w.assign(
    month=val_w["target_date"].dt.month,
    weather_error=lambda d: (
        d["target"] - d["weather_pred"]
    ).abs(),
    weather_bias=lambda d: (
        d["target"] - d["weather_pred"]
    ),
)

weather_monthly = weather_diagnostics.groupby("month").agg(
    weather_mae=("weather_error", "mean"),
    weather_bias=("weather_bias", "mean"),
    avg_prediction=("weather_pred", "mean"),
)

weather_monthly["original_xgb_mae"] = monthly_errors["xgb_mae"]
weather_monthly["naive_mae"] = monthly_errors["naive_mae"]

display(weather_monthly.round(2))

,weather_mae,weather_bias,avg_prediction,original_xgb_mae,naive_mae
month,,,,,
1,4.67,2.24,86.430000,5.07,5.09
2,2.01,0.47,74.559998,2.30,4.15
3,3.40,0.61,70.500000,4.14,3.97
4,5.07,3.75,91.010002,5.19,6.79
5,20.77,19.13,116.629997,19.95,12.79
6,36.96,36.94,114.860001,22.02,12.44
7,13.93,10.69,130.289993,13.10,11.47
8,6.08,-2.16,129.710007,8.47,10.19
9,10.59,5.16,113.580002,10.39,12.78


Residual XGBoost

In [20]:
residual_features = features_weather.copy()

# Weekly seasonal baseline for each forecast horizon
residual_features["naive_pred"] = [
    row[f"lag_{8 - int(row['horizon'])}"]
    for _, row in residual_features.iterrows()
]

residual_features["residual_target"] = (
    residual_features["target"]
    - residual_features["naive_pred"]
)

train_r = residual_features[
    residual_features["split"] == "train"
]

val_r = residual_features[
    residual_features["split"] == "validation"
].copy()

residual_model = XGBRegressor(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    tree_method="hist",
    device="cpu",
    random_state=42,
    n_jobs=-1,
)

residual_model.fit(
    train_r[weather_feature_cols],
    train_r["residual_target"]
)

val_r["residual_pred"] = residual_model.predict(
    val_r[weather_feature_cols]
)

val_r["final_pred"] = (
    val_r["naive_pred"] + val_r["residual_pred"]
)

print(
    "Residual weather XGB MAE:",
    round(mean_absolute_error(
        val_r["target"], val_r["final_pred"]
    ), 3)
)

print(
    "Residual weather XGB RMSE:",
    round(root_mean_squared_error(
        val_r["target"], val_r["final_pred"]
    ), 3)
)

Residual weather XGB MAE: 8.611
Residual weather XGB RMSE: 12.515


In [21]:
residual_no_weather = XGBRegressor(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    tree_method="hist",
    device="cpu",
    random_state=42,
    n_jobs=-1,
)

residual_no_weather.fit(
    train_r[feature_cols],
    train_r["residual_target"]
)

val_r["no_weather_pred"] = (
    val_r["naive_pred"]
    + residual_no_weather.predict(val_r[feature_cols])
)

print(
    "Residual XGB without weather MAE:",
    round(mean_absolute_error(
        val_r["target"], val_r["no_weather_pred"]
    ), 3)
)

print(
    "Residual XGB without weather RMSE:",
    round(root_mean_squared_error(
        val_r["target"], val_r["no_weather_pred"]
    ), 3)
)

Residual XGB without weather MAE: 8.596
Residual XGB without weather RMSE: 12.277


In [22]:
import numpy as np
import pandas as pd

energy = v2["energy_met_mu"]

CONTEXT = 56
HORIZON = 7

contexts, origins, actuals = [], [], []

for i in range(CONTEXT, len(energy) - HORIZON + 1):
    history = energy.iloc[i-CONTEXT:i]
    future = energy.iloc[i:i+HORIZON]

    if history.isna().any() or future.isna().any():
        continue

    if not (
        pd.Timestamp("2024-01-01") <= future.index[0]
        and future.index[-1] < pd.Timestamp("2025-01-01")
    ):
        continue

    contexts.append(history.to_numpy(dtype=np.float32))
    origins.append(history.index[-1])
    actuals.append(future.to_numpy(dtype=np.float32))

print("Validation origins:", len(origins))
print("Context array:", np.asarray(contexts).shape)
print("Actual array:", np.asarray(actuals).shape)

Validation origins: 360
Context array: (360, 56)
Actual array: (360, 7)


In [24]:
import torch
import timesfm as tfm
from chronos import BaseChronosPipeline

# Chronos-Bolt Mini
chronos_model = BaseChronosPipeline.from_pretrained(
    "amazon/chronos-bolt-mini",
    device_map="cpu",
    torch_dtype=torch.float32,
)

# TimesFM 2.5
timesfm_model = tfm.TimesFM_2p5_200M_torch.from_pretrained(
    "google/timesfm-2.5-200m-pytorch"
)

timesfm_model.compile(
    tfm.ForecastConfig(
        max_context=28,
        max_horizon=7,
        normalize_inputs=True,
        per_core_batch_size=8,
        use_continuous_quantile_head=True,
        force_flip_invariance=True,
        infer_is_positive=True,
        fix_quantile_crossing=True,
    )
)

print("Chronos and TimesFM loaded successfully.")

Chronos and TimesFM loaded successfully.


In [25]:
import numpy as np
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

# Assumes contexts and actuals are prepared from the previous cell
assert np.asarray(contexts).shape == (360, 56)
assert np.asarray(actuals).shape == (360, 7)

forecast, _ = timesfm_model.forecast(
    horizon=7,
    inputs=contexts,
)

timesfm_predictions = np.asarray(forecast)

assert timesfm_predictions.shape == (360, 7)

print(
    "TimesFM MAE:",
    round(mean_absolute_error(
        np.asarray(actuals).ravel(),
        timesfm_predictions.ravel()
    ), 3)
)

print(
    "TimesFM RMSE:",
    round(root_mean_squared_error(
        np.asarray(actuals).ravel(),
        timesfm_predictions.ravel()
    ), 3)
)

TimesFM MAE: 7.156
TimesFM RMSE: 9.871


TimesFM errors by month

In [26]:
timesfm_eval = pd.DataFrame({
    "origin": pd.to_datetime(origins).repeat(7),
    "horizon": np.tile(np.arange(1, 8), len(origins)),
    "actual": np.asarray(actuals).reshape(-1),
    "timesfm_pred": timesfm_predictions.reshape(-1),
})

timesfm_eval["target_date"] = (
    timesfm_eval["origin"]
    + pd.to_timedelta(timesfm_eval["horizon"], unit="D")
)

timesfm_eval["month"] = timesfm_eval["target_date"].dt.month
timesfm_eval["error"] = (
    timesfm_eval["actual"] - timesfm_eval["timesfm_pred"]
)

timesfm_monthly = timesfm_eval.groupby("month").agg(
    timesfm_mae=("error", lambda x: x.abs().mean()),
    timesfm_bias=("error", "mean"),
    actual_mean=("actual", "mean"),
    predicted_mean=("timesfm_pred", "mean"),
)

timesfm_monthly["naive_mae"] = monthly_errors["naive_mae"]
timesfm_monthly["xgb_mae"] = monthly_errors["xgb_mae"]

display(timesfm_monthly.round(2))

,timesfm_mae,timesfm_bias,actual_mean,predicted_mean,naive_mae,xgb_mae
month,,,,,,
1,4.44,0.81,88.669998,87.860001,5.09,5.07
2,4.41,-4.28,75.029999,79.309998,4.15,2.30
3,3.46,0.82,71.099998,70.279999,3.97,4.14
4,5.21,4.33,94.760002,90.430000,6.79,5.19
5,12.75,10.55,135.759995,125.220001,12.79,19.95
6,10.83,-3.61,151.800003,155.410004,12.44,22.02
7,10.01,-2.09,140.979996,143.070007,11.47,13.10
8,7.30,-3.88,127.550003,131.429993,10.19,8.47
9,10.16,-2.82,118.739998,121.559998,12.78,10.39


In [27]:
import numpy as np
import torch

from chronos import BaseChronosPipeline
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

chronos_model = BaseChronosPipeline.from_pretrained(
    "amazon/chronos-bolt-mini",
    device_map="cpu",
    torch_dtype=torch.float32,
)

predictions = []

for context in contexts:
    quantiles, _ = chronos_model.predict_quantiles(
        inputs=torch.tensor(context).unsqueeze(0),
        prediction_length=7,
        quantile_levels=[0.1, 0.5, 0.9],
    )

    # Median (P50) forecast
    predictions.append(
        quantiles[0, :, 1].detach().cpu().numpy()
    )

chronos_predictions = np.asarray(predictions)

print(
    "Chronos MAE:",
    round(mean_absolute_error(
        np.asarray(actuals).ravel(),
        chronos_predictions.ravel()
    ), 3)
)

print(
    "Chronos RMSE:",
    round(root_mean_squared_error(
        np.asarray(actuals).ravel(),
        chronos_predictions.ravel()
    ), 3)
)

Chronos MAE: 7.181
Chronos RMSE: 9.817


In [28]:
xgb_pairs = set(
    zip(
        val["origin"],
        val["horizon"]
    )
)

foundation_pairs = set(
    zip(
        timesfm_eval["origin"],
        timesfm_eval["horizon"]
    )
)

print("XGBoost rows:", len(xgb_pairs))
print("Foundation rows:", len(foundation_pairs))
print("Identical evaluation pairs:", xgb_pairs == foundation_pairs)

XGBoost rows: 2520
Foundation rows: 2520
Identical evaluation pairs: True


In [29]:
# Both models use the same 360 origins and ordering.
assert timesfm_predictions.shape == chronos_predictions.shape == (360, 7)

horizon_comparison = pd.DataFrame({
    "horizon": np.arange(1, 8),
    "timesfm_mae": np.abs(
        np.asarray(actuals) - timesfm_predictions
    ).mean(axis=0),
    "chronos_mae": np.abs(
        np.asarray(actuals) - chronos_predictions
    ).mean(axis=0),
})

horizon_comparison["difference"] = (
    horizon_comparison["chronos_mae"]
    - horizon_comparison["timesfm_mae"]
)

display(horizon_comparison.round(3))

,horizon,timesfm_mae,chronos_mae,difference
0,1,3.988,4.142,0.155
1,2,5.731,6.037,0.306
2,3,6.746,6.859,0.113
3,4,7.417,7.431,0.014
4,5,8.143,8.128,-0.015
5,6,8.742,8.594,-0.148
6,7,9.326,9.073,-0.254


In [30]:
import numpy as np

actual = np.asarray(actuals)
tf_errors = np.abs(actual - timesfm_predictions)
ch_errors = np.abs(actual - chronos_predictions)

# Positive = TimesFM has lower MAE
daily_advantage = (ch_errors - tf_errors).mean(axis=1)

rng = np.random.default_rng(42)
B = 2000
BLOCK = 14
n = len(daily_advantage)

boot = np.empty(B)

for b in range(B):
    sampled = []

    while len(sampled) < n:
        start = rng.integers(0, n - BLOCK + 1)
        sampled.extend(range(start, start + BLOCK))

    boot[b] = daily_advantage[np.array(sampled[:n])].mean()

print("TimesFM advantage (MU):", round(daily_advantage.mean(), 3))
print("95% bootstrap interval:", np.quantile(boot, [0.025, 0.975]))

TimesFM advantage (MU): 0.024
95% bootstrap interval: [-0.55844949  0.5652082 ]
